# Dignity — train the risk head

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crichalchemist/Dignity/blob/master/notebooks/train_risk.ipynb)

This notebook trains one Dignity model end to end on the synthetic transaction data. It is the
same path `dignity-train` runs, with the intermediate pieces visible:

1. load a YAML config and pick the device
2. generate the data, run the privacy stage once, split at the sequence level
3. build `Dignity(task=...)`, train with `train_epoch` / `validate_epoch`, keep the best checkpoint
4. plot the curves, score the validation windows, look at the attention weights
5. export the best checkpoint to ONNX

A GPU runtime is recommended: `config/train_risk.yaml` takes about 2.5 minutes per epoch on an
8-core CPU and well under a minute on a T4. On Colab the setup cell clones the repo and installs
only the ONNX packages Colab lacks (everything else is preinstalled) and mounts Google Drive so
checkpoints and results outlive the runtime (`MOUNT_DRIVE = False` skips that).

In [ ]:
import os
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path

REPO = "https://github.com/crichalchemist/Dignity"
IN_COLAB = "google.colab" in sys.modules
MOUNT_DRIVE = True  # Colab only: keep checkpoints and results in Google Drive


def find_root(start: Path) -> Path | None:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "core").is_dir():
            return candidate
    return None


root = find_root(Path.cwd())
if root is None and IN_COLAB and Path("/content/Dignity/pyproject.toml").exists():
    # Restarting the runtime resets the working directory to /content but keeps
    # the clone from the earlier run; cloning again over it would fail.
    root = Path("/content/Dignity")
if root is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    root = Path.cwd() / "Dignity"
elif IN_COLAB and root == Path("/content/Dignity"):
    # A clone left over from an earlier run of this runtime: bring it up to date.
    # Python caches imported modules, so after a pull restart the runtime before
    # re-running the cells below, or the old code keeps running.
    subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=True)
os.chdir(root)
sys.path.insert(0, str(root))
print("repo root:", root)

if IN_COLAB:
    # Colab already ships torch, numpy, pandas, scipy, scikit-learn and matplotlib.
    # Do not `pip install -r requirements.txt` here: its numpy<2 pin makes pip try
    # to downgrade numpy underneath Colab's preinstalled stack, which runs for many
    # minutes and looks like a hang. Install only what is actually missing.
    needed = {"onnx": "onnx", "onnxruntime": "onnxruntime", "yaml": "pyyaml"}
    missing = [pkg for mod, pkg in needed.items() if find_spec(mod) is None]
    if missing:
        print("installing:", " ".join(missing))
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *missing], check=True
        )
    if MOUNT_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")

## Configuration

Everything comes from one YAML file, exactly as on the command line. The four knobs below are
the ones worth changing from a notebook:

- `CONFIG`: `config/train_risk.yaml` is the shipped risk setup with the privacy stage on;
  `config/colab.yaml` is a smaller model with shorter windows for quick runs.
- `EPOCHS` and `CHECKPOINT_DIR` override the YAML when set. On Colab the setup cell mounts
  Google Drive and checkpoints default to `MyDrive/dignity/checkpoints/<config>` (with a
  `_noprivacy` suffix when `PRIVACY = False`), so a run survives its runtime.
- `PRIVACY = False` drops the `privacy:` block. Run once with it on and once off to measure what
  the noise costs the model; the shipped noise scales are large on purpose (see
  `docs/PRIVACY.md`).

In [ ]:
import torch

from core.config import DignityConfig

CONFIG = "config/train_risk.yaml"
EPOCHS = None  # None keeps the YAML value
CHECKPOINT_DIR = None  # None: Google Drive when mounted (Colab), else the YAML value
PRIVACY = True  # False trains a no-privacy baseline for comparison

config = DignityConfig.from_yaml(CONFIG)
if EPOCHS is not None:
    config.train.epochs = EPOCHS
if CHECKPOINT_DIR is None and Path("/content/drive/MyDrive").is_dir():
    # Runtime storage disappears with the runtime: keep every run's checkpoints,
    # history and metrics in Drive, one folder per config and privacy setting.
    CHECKPOINT_DIR = "/content/drive/MyDrive/dignity/checkpoints/" + Path(CONFIG).stem
    if not PRIVACY:
        CHECKPOINT_DIR += "_noprivacy"
if CHECKPOINT_DIR is not None:
    config.train.checkpoint_dir = CHECKPOINT_DIR
if not PRIVACY:
    config.privacy = None

device = torch.device(config.device if torch.cuda.is_available() else "cpu")
use_amp = config.train.use_amp and device.type == "cuda"
torch.manual_seed(config.seed)

print(f"task={config.model.task}  epochs={config.train.epochs}  device={device}")
print("checkpoints:", config.train.checkpoint_dir)
if config.privacy is None:
    print("privacy: off")
else:
    print(
        f"privacy: ε_total={config.privacy.epsilon_total}, "
        f"k={config.privacy.k}, columns={list(config.privacy.features)}"
    )

## Data

`SyntheticGenerator.generate_dataset` writes 800 normal and 200 anomalous sequences end to end
as one frame, each `seq_len + 20` rows long. `process_blocks` treats every such block as one
independent sequence: the privacy stage runs once on the whole frame, blocks are shuffled and
split into train and validation, the scaler is fit on the training blocks only, and signals and
sliding windows are computed inside each block so nothing crosses a joint.

In [ ]:
import numpy as np

from data.loader import create_dataloader
from data.pipeline import TransactionPipeline
from data.source.synthetic import SyntheticGenerator

block_len = config.data.seq_len + 20  # one generated sequence; 21 windows each
df = SyntheticGenerator(seed=config.seed).generate_dataset(
    num_normal=800, num_anomalous=200, seq_len=block_len
)

pipeline = TransactionPipeline(
    seq_len=config.data.seq_len,
    features=config.data.features,
    privacy=config.privacy,
)
(X_train, y_train), (X_val, y_val) = pipeline.process_blocks(
    df.drop(columns="label"),
    df["label"].to_numpy(),
    block_len=block_len,
    test_size=config.data.test_size,
    rng=np.random.default_rng(config.seed),
)

if pipeline.privacy_manager is not None:
    budget = pipeline.privacy_manager.budget
    print(f"privacy: ε spent {budget.spent:.3f} of {budget.epsilon_total:.3f}")
print("features used:", pipeline.available_features)
print(
    f"train windows {len(X_train)} ({y_train.mean():.1%} anomalous), "
    f"val windows {len(X_val)} ({y_val.mean():.1%} anomalous)"
)

train_loader = create_dataloader(
    X_train, y_train, batch_size=config.data.batch_size, shuffle=True, device=device
)
val_loader = create_dataloader(
    X_val, y_val, batch_size=config.data.batch_size, shuffle=False, device=device
)

## Model

`Dignity(task=...)` is the only model entry point: the CNN → LSTM → attention backbone plus one
head. The input width is whatever the pipeline actually produced, not `model.input_size` from
the YAML (configured features that the data cannot provide are dropped).

In [ ]:
import torch.nn as nn

from models.dignity import Dignity

model = Dignity(
    task=config.model.task,
    input_size=len(pipeline.available_features),
    hidden_size=config.model.hidden_size,
    n_layers=config.model.n_layers,
    dropout=config.model.dropout,
).to(device)
print(model.summary())

optimizer = torch.optim.AdamW(
    model.parameters(), lr=config.train.lr, weight_decay=config.train.weight_decay
)
criterion = {"risk": nn.BCELoss(), "forecast": nn.MSELoss()}.get(
    config.model.task, nn.CrossEntropyLoss()
)

## Training

One line per epoch. The best validation loss is saved to `dignity_<task>_best.pt` in the
checkpoint directory, and a numbered checkpoint every `save_interval` epochs. `history.json` with the per-epoch curves
lands next to them, and the validation cell writes `metrics.json`.

In [ ]:
import json

from train.engine import save_checkpoint, train_epoch, validate_epoch

ckpt_dir = Path(config.train.checkpoint_dir)
ckpt_dir.mkdir(parents=True, exist_ok=True)
best_path = ckpt_dir / f"dignity_{config.model.task}_best.pt"

history = {"train_loss": [], "val_loss": [], "val_accuracy": []}
best_val = float("inf")
for epoch in range(1, config.train.epochs + 1):
    train_metrics = train_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        use_amp=use_amp,
        grad_clip=config.train.gradient_clip,
        log_interval=config.train.log_interval,
    )
    val_metrics = validate_epoch(
        model=model, dataloader=val_loader, criterion=criterion, device=device
    )
    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_accuracy"].append(val_metrics.get("accuracy", float("nan")))

    line = f"epoch {epoch:3d}  train {train_metrics['loss']:.4f}  val {val_metrics['loss']:.4f}"
    if "accuracy" in val_metrics:
        line += f"  acc@0.5 {val_metrics['accuracy']:.3f}"
    metrics = {"train": train_metrics, "val": val_metrics}
    if epoch % config.train.save_interval == 0:
        save_checkpoint(
            model,
            optimizer,
            epoch,
            metrics,
            str(ckpt_dir / f"dignity_{config.model.task}_epoch{epoch}.pt"),
        )
    if val_metrics["loss"] < best_val:
        best_val = val_metrics["loss"]
        save_checkpoint(model, optimizer, epoch, metrics, str(best_path))
        line += "  *"
    print(line)
print(f"best val loss {best_val:.4f} → {best_path}")
(ckpt_dir / "history.json").write_text(json.dumps(history, indent=1))

## Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 3.5))
ax_loss.plot(epochs, history["train_loss"], label="train")
ax_loss.plot(epochs, history["val_loss"], label="val")
ax_loss.set(xlabel="epoch", ylabel="loss", title="loss")
ax_loss.legend()
ax_acc.plot(epochs, history["val_accuracy"], color="tab:green")
ax_acc.set(xlabel="epoch", ylabel="accuracy @ 0.5", title="validation", ylim=(0, 1))
plt.tight_layout()
plt.show()

## Validation scores

Reload the best checkpoint and score every validation window. Accuracy at a fixed 0.5
threshold is what the training loop reports; ROC AUC is threshold-free and the better
single number for an anomaly detector. The validation loader is not shuffled, so the scores
line up with `y_val`.

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)

from train.engine import load_checkpoint

load_checkpoint(model, None, str(best_path), device)
model.eval()
with torch.no_grad():
    scores = torch.cat(
        [
            model.predict(x.to(device)).reshape(len(x), -1)[:, 0].cpu()
            for x, _ in val_loader
        ]
    ).numpy()

if config.model.task == "risk":
    labels = y_val.astype(int)
    predicted = (scores > 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predicted, average="binary", zero_division=0
    )
    results = {
        "roc_auc": float(roc_auc_score(labels, scores)),
        "precision_at_0.5": float(precision),
        "recall_at_0.5": float(recall),
        "f1_at_0.5": float(f1),
        "confusion_matrix": confusion_matrix(labels, predicted).tolist(),
    }
    (ckpt_dir / "metrics.json").write_text(json.dumps(results, indent=1))
    print(f"ROC AUC          {results['roc_auc']:.4f}")
    print(f"precision @ 0.5  {precision:.3f}")
    print(f"recall @ 0.5     {recall:.3f}")
    print(f"F1 @ 0.5         {f1:.3f}")
    print("confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(labels, predicted))
else:
    print(f"task={config.model.task}: mean prediction {scores.mean():.4f}")

## Where the model looked

The backbone returns additive-attention weights over the window along with the prediction.
One normal and one anomalous validation window, with the scaled `volume` feature behind the
attention profile.

In [ ]:
feature = "volume" if "volume" in pipeline.available_features else None
column = pipeline.available_features.index(feature) if feature else 0
examples = [
    ("normal", int(np.flatnonzero(y_val == 0)[0])),
    ("anomalous", int(np.flatnonzero(y_val == 1)[0])),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, (name, idx) in zip(axes, examples, strict=True):
    x = torch.as_tensor(X_val[idx : idx + 1], dtype=torch.float32, device=device)
    with torch.no_grad():
        prediction, attention = model(x)
    attention = attention[0].cpu().numpy()
    ax.fill_between(range(len(attention)), attention, color="tab:red", alpha=0.35)
    ax.set(xlabel="step", ylabel="attention weight")
    ax.set_title(f"{name} window — score {float(prediction.reshape(-1)[0]):.2f}")
    twin = ax.twinx()
    twin.plot(X_val[idx, :, column], color="tab:gray", linewidth=1)
    twin.set_ylabel(f"{pipeline.available_features[column]} (scaled)")
plt.tight_layout()
plt.show()

## Export to ONNX

Same export the CLI does (`python -m export.to_onnx`). The exporter runs the model on CPU and
verifies the ONNX graph against PyTorch before returning.

In [ ]:
EXPORT = True

if EXPORT:
    from export.to_onnx import export_to_onnx

    onnx_path = ckpt_dir / f"dignity_{config.model.task}.onnx"
    export_to_onnx(
        model.cpu(),
        str(onnx_path),
        input_shape=(1, config.data.seq_len, len(pipeline.available_features)),
    )
    model.to(device)
    print("wrote", onnx_path)

## Notes

- Everything here is synthetic data from `data/source/synthetic.py`; the numbers say the
  pipeline and model work, not that they detect anything real.
- To measure the privacy cost, run the notebook twice with `PRIVACY` on and off and compare
  the ROC AUC. The shipped Laplace scales (2000 for `volume`, 1000 for `price`) exercise the
  stage rather than tune it; `docs/PRIVACY.md` explains what each mechanism does and does not
  guarantee.
- The equivalent command line is `dignity-train --config <CONFIG>`; it writes the same
  checkpoints to `train.checkpoint_dir`.